# CLO Email Extraction — Interactive GUI

**How to use:**
1. **Run Cell 1** (Setup — loads extractor, queue, field mapping)
2. **Run Cell 2** (Interactive GUI — runs right here in the notebook)
3. Process emails: review extracted fields, edit if needed, Accept or Skip
4. (Optional) Cell 3 launches a web-based UI if you prefer a browser

**Later cells:** View saved data, check training status, fine-tune the model.

In [1]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 1: SETUP — Load model, queue, and widget library
# ═══════════════════════════════════════════════════════════════════════
# Run this ONCE when you start or restart the kernel.
# ═══════════════════════════════════════════════════════════════════════

import sys, os, json
from datetime import datetime
from IPython.display import display, HTML, Markdown, clear_output
import ipywidgets as widgets

ROOT = os.getcwd()
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

# Configure extractor: "nuextract" = local model, "anthropic" = API, "bedrock" = AWS, "rules" = no LLM
os.environ.setdefault("CLO_EXTRACTOR", "rules")
# Device: "mps" = Mac GPU, "cuda" = NVIDIA GPU, "cpu" = no GPU
os.environ.setdefault("NUEXTRACT_DEVICE", "cuda")

# Force-reload so config changes take effect in Jupyter
import importlib
import backend.config;  importlib.reload(backend.config)
import backend.field_mapping;  importlib.reload(backend.field_mapping)
import backend.extractor;  importlib.reload(backend.extractor)

from backend.watcher import EmailQueue
from backend.extractor import get_extractor, detect_email_type, format_tranche_pricing, parse_deal_fields
from backend.models import ExtractionResult, new_id, now_iso
from backend import store, config
from backend.field_mapping import (
    load_mapping, get_form_fields, build_store_record, get_active_fields,
    STORE_DEALS, STORE_TRANSACTIONS, STORE_MANAGERS, STORE_DEAL_ORDERS,
)

queue = EmailQueue()
queue.scan()

# Load field mapping (from data/field_mapping.json or defaults)
_field_mapping = load_mapping()

print("Loading extractor (first run downloads ~7GB model)...")
try:
    extractor = get_extractor()
    print(f"✅ Extractor ready: {type(extractor).__name__}")
except Exception as e:
    extractor = None
    print(f"⚠️  No LLM extractor ({e}), will use heuristic fallback")

print(f"📬 {queue.size()} email(s) in queue: {queue.remaining()}")

# Show field mapping summary
_active = get_active_fields(_field_mapping)
_custom = [f for f in _active if f["name"] not in ExtractionResult.model_fields]
print(f"🗂️  {len(_active)} active extraction fields" + (f" ({len(_custom)} custom)" if _custom else ""))
if _custom:
    for f in _custom:
        print(f"    + {f['name']} -> {f['store']}.{f['store_field']}")

Loading extractor (first run downloads ~7GB model)...
[RuleBased] Custom rules module loaded: my_rules_plus.py
[RuleBased] Extractor ready (no LLM required)
[FieldMapping] Skipping 5 fields: clo-deals__Deal Documents, clo-managers__Ultimate Parent, clo-managers__CRD Number, clo-managers__SEC Number, clo-managers__Website
✅ Extractor ready: _MappingAwareExtractor
📬 51 email(s) in queue: ['2026-03-21_181152__clo_refi__allegro_clo_xvi,_ltd._--_announcement_&_roller_process_(144a_regs)(external)__other__.txt', '2026-03-21_181342__clo_refinancing__aimco_clo_series_2018-b___3nc1_announcement___BNP__.txt', '2026-03-21_181356__clo_primary__announcing_carlyle_2024-1_reset___Citi__.txt', '2026-03-21_181657__announcement_-_elmwood_clo_28_partial_refi_`__other__.txt', '2026-03-21_181711__clo_primary__announcing_1988_clo_4_reset_(muzinich)___Citi__.txt', '2026-03-21_181725__clo_new_issue__bcred_pc_static_clo_2026-1___eurrc,__1.6y_aaa_wal_announcement___BNP__.txt', '2026-03-21_181754__clo_refinancin

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 2: INTERACTIVE GUI — Side-by-side layout (runs in notebook)
# ═══════════════════════════════════════════════════════════════════════
#   LEFT panel:  Email preview (scrollable HTML)
#   RIGHT panel: Editable extraction form + buttons
#
#   No web server needed — runs entirely inside VS Code / Jupyter.
#   FORM_FIELDS is generated from data/field_mapping.json so that
#   custom fields you add via configure_fields.ipynb appear automatically.
# ═══════════════════════════════════════════════════════════════════════

# Build form fields from the field mapping config
FORM_FIELDS = get_form_fields(_field_mapping)

_gui_state = {"file": None, "html": None, "extraction": None}

# ── Status bar (top) ──
status_label = widgets.HTML(
    value="<b>Initializing...</b>",
    layout=widgets.Layout(padding="6px 10px"),
)

# ── LEFT panel: Email preview ──
email_html = widgets.HTML(
    value="<p style='color:#888; padding:20px;'><i>Run to load first email...</i></p>",
    layout=widgets.Layout(overflow_y="auto"),
)
left_panel = widgets.VBox(
    [widgets.HTML(value="<b style='font-size:13px;'>📧 Email Preview</b>"), email_html],
    layout=widgets.Layout(
        width="50%", min_width="300px",
        border="1px solid #666",
        padding="8px",
        height="900px",
        overflow_y="auto",
    ),
)

# ── RIGHT panel: Form fields ──
field_widgets = {}
form_rows = []
for fname, label, ftype in FORM_FIELDS:
    lbl = widgets.Label(value=label, layout=widgets.Layout(width="160px", min_width="160px"))
    if ftype == "dropdown":
        w = widgets.Dropdown(
            options=["announced", "updated", "priced"],
            value="announced",
            layout=widgets.Layout(width="auto", flex="1 1 auto"),
        )
    elif ftype == "float":
        w = widgets.Text(value="", placeholder="e.g. 500.0", layout=widgets.Layout(width="auto", flex="1 1 auto"))
    elif ftype == "textarea":
        w = widgets.Textarea(
            value="",
            layout=widgets.Layout(width="auto", flex="1 1 auto", height="130px"),
        )
    else:
        w = widgets.Text(value="", layout=widgets.Layout(width="auto", flex="1 1 auto"))
    field_widgets[fname] = w
    form_rows.append(widgets.HBox([lbl, w], layout=widgets.Layout(margin="2px 0")))

form_scroll = widgets.VBox(
    form_rows,
    layout=widgets.Layout(overflow_y="auto", flex="1 1 auto"),
)

# Buttons
btn_accept = widgets.Button(description="Accept & Save", button_style="success", icon="check",
                            layout=widgets.Layout(width="auto", flex="1 1 auto"))
btn_skip = widgets.Button(description="Skip", button_style="warning", icon="forward",
                          layout=widgets.Layout(width="auto", flex="1 1 auto"))
btn_refresh = widgets.Button(description="Refresh", button_style="info", icon="refresh",
                             layout=widgets.Layout(width="auto", flex="1 1 auto"))
button_bar = widgets.HBox([btn_accept, btn_skip, btn_refresh],
                          layout=widgets.Layout(margin="6px 0", gap="6px"))

right_panel = widgets.VBox(
    [
        widgets.HTML(value="<b style='font-size:13px;'>Extracted Fields</b> "
                     "<span style='color:#888; font-size:11px;'>(edit before accepting)</span>"),
        form_scroll,
        button_bar,
    ],
    layout=widgets.Layout(
        width="50%", min_width="300px",
        border="1px solid #666",
        padding="8px",
        height="900px",
    ),
)

# ── Bottom: Log output ──
log_output = widgets.Output(layout=widgets.Layout(
    border="1px solid #555", padding="6px",
    max_height="150px", overflow_y="auto", width="100%",
))

# ── Done prompt (shown when queue is empty) ──
done_label = widgets.HTML(
    value="<h3 style='color:#66bb6a;'>✅ All Emails Processed</h3>"
          "<p style='color:#a0a0b8; font-size:13px;'>The email queue is empty. "
          "Drop more .txt files into data/inbox/ and click Refresh.</p>",
)
btn_refresh_done = widgets.Button(description="Refresh Queue", button_style="info", icon="refresh",
                                  layout=widgets.Layout(width="auto"))
done_box = widgets.VBox(
    [done_label, btn_refresh_done],
    layout=widgets.Layout(
        display="none",
        border="2px solid #66bb6a",
        border_radius="12px",
        padding="24px",
        width="100%",
        align_items="center",
    ),
)

# ── Layout assembly ──
main_panels = widgets.HBox(
    [left_panel, right_panel],
    layout=widgets.Layout(width="100%", gap="8px"),
)
gui = widgets.VBox(
    [status_label, done_box, main_panels, log_output],
    layout=widgets.Layout(width="100%"),
)


# ═══════════════════════════════════════════════════════════════════════
# Event handlers
# ═══════════════════════════════════════════════════════════════════════

def _update_status():
    f = _gui_state["file"]
    n = queue.size()
    if f:
        status_label.value = (
            f"<b style='font-size:14px;'>📧 {f}</b>"
            f"&nbsp;&nbsp;|&nbsp;&nbsp;{n} email(s) in queue"
        )
    else:
        status_label.value = "<b style='font-size:14px;'>✅ Queue empty — all emails processed!</b>"


def _show_done_prompt():
    done_box.layout.display = "flex"
    main_panels.layout.display = "none"


def _hide_done_prompt():
    done_box.layout.display = "none"
    main_panels.layout.display = "flex"


def _load_next_email():
    queue.scan()
    fname = queue.peek()
    _gui_state["file"] = fname
    _gui_state["extraction"] = None
    _gui_state["html"] = None

    if not fname:
        email_html.value = "<p style='color:#888; text-align:center; padding:40px;'><i>No emails to process.</i></p>"
        for w in field_widgets.values():
            if isinstance(w, widgets.Dropdown):
                w.value = "announced"
            else:
                w.value = ""
        _update_status()
        _show_done_prompt()
        return

    _hide_done_prompt()
    html_content = queue.read_file(fname)
    _gui_state["html"] = html_content
    email_html.value = html_content

    with log_output:
        print(f"Extracting from {fname}...")

    extraction = None
    try:
        if extractor:
            result = extractor.extract(html_content)
            extraction = result.model_dump()
            with log_output:
                print(f"✅ Extraction complete for {fname}")
        else:
            extraction = ExtractionResult(email_type=detect_email_type(html_content)).model_dump()
            with log_output:
                print(f"⚠️  Heuristic fallback for {fname}")
    except Exception as e:
        extraction = ExtractionResult(email_type=detect_email_type(html_content)).model_dump()
        with log_output:
            print(f"❌ Extraction error: {e}")

    # Parse deal fields + tranche pricing directly from HTML tables
    deal_fields = parse_deal_fields(html_content)
    for key, val in deal_fields.items():
        if val and not extraction.get(key):
            extraction[key] = val
    pricing = format_tranche_pricing(html_content)
    for key in ("ipt", "updated_guidance", "final_pricing"):
        if pricing.get(key):
            extraction[key] = pricing[key]

    _gui_state["extraction"] = extraction

    for fname_key, _, ftype in FORM_FIELDS:
        val = extraction.get(fname_key)
        w = field_widgets[fname_key]
        if ftype == "dropdown":
            w.value = val if val in ["announced", "updated", "priced"] else "announced"
        elif ftype == "float":
            w.value = str(val) if val is not None else ""
        elif ftype == "textarea":
            w.value = str(val) if val else ""
        else:
            w.value = str(val) if val else ""

    _update_status()


def _get_form_values() -> dict:
    result = {}
    for fname, _, ftype in FORM_FIELDS:
        w = field_widgets[fname]
        if ftype == "dropdown":
            result[fname] = w.value
        elif ftype == "float":
            txt = w.value.strip()
            if txt:
                try:
                    result[fname] = float(txt)
                except ValueError:
                    result[fname] = None
            else:
                result[fname] = None
        elif ftype == "textarea":
            result[fname] = w.value.strip() or None
        else:
            result[fname] = w.value.strip() or None
    return result


def _on_accept(btn):
    if not _gui_state["file"]:
        with log_output:
            print("No email loaded.")
        return

    extraction = _get_form_values()
    email_type = extraction.get("email_type") or "announced"
    html_content = _gui_state["html"]
    current_file = _gui_state["file"]
    now = datetime.utcnow().isoformat()
    today = datetime.utcnow().strftime("%Y-%m-%d")

    # Save training example for future AI fine-tuning
    count = store.append_training_example(html_content, extraction)

    # Build store records from field mapping (produces SharePoint field names)
    deal_data = build_store_record(extraction, STORE_DEALS, _field_mapping)
    txn_data = build_store_record(extraction, STORE_TRANSACTIONS, _field_mapping)
    mgr_data = build_store_record(extraction, STORE_MANAGERS, _field_mapping)
    order_data = build_store_record(extraction, STORE_DEAL_ORDERS, _field_mapping)

    # Get deal title for matching (clo-deals.json uses "Title", not "deal_name")
    deal_title = deal_data.get("Title") or extraction.get("deal_name") or ""

    if email_type == "announced":
        if deal_data:
            existing = store.find_record(store.DEALS, lambda r: r.get("Title", "").strip() == deal_title.strip())
            if existing:
                store.update_record(store.DEALS, lambda r: r.get("Title", "").strip() == deal_title.strip(), deal_data)
            else:
                store.append_record(store.DEALS, deal_data)

        if txn_data:
            txn_data.setdefault("Deal", deal_title)
            txn_data.setdefault("Title", deal_title)
            txn_data.setdefault("Status", "Announced")
            txn_data.setdefault("Announcement Date", extraction.get("announced_date") or extraction.get("clo-transactions__Announcement Date") or today)
            store.append_record(store.TRANSACTIONS, txn_data)

        # Create manager if new
        mgr_name = mgr_data.get("Name") or extraction.get("collateral_manager_legal_entity") or ""
        mgr_short = mgr_data.get("Short Name") or extraction.get("collateral_manager_short") or ""
        if mgr_name and mgr_short:
            existing_mgr = store.find_record(store.MANAGERS, lambda r: r.get("Short Name", "").strip().lower() == mgr_short.strip().lower())
            if not existing_mgr:
                mgr_record = dict(mgr_data)
                mgr_record.setdefault("Name", mgr_name)
                mgr_record.setdefault("Short Name", mgr_short)
                store.append_record(store.MANAGERS, mgr_record)

        if order_data:
            order_data.setdefault("deal_name", deal_title)
            order_data.setdefault("created_at", now)
            order_data["updated_at"] = now
            store.append_record(store.DEAL_ORDERS, order_data)

        with log_output:
            print(f"\u2705 Accepted {current_file} — saved deal '{deal_title}' + transaction (training #{count})")

    elif email_type == "updated":
        update_fields = {"Status": "Announced"}  # Updated guidance keeps Announced status
        update_fields.update(txn_data)
        store.update_record(
            store.TRANSACTIONS,
            lambda r: r.get("Deal", "").strip() == deal_title.strip() and r.get("Status") != "Priced",
            update_fields,
        )
        if deal_data:
            store.update_record(store.DEALS, lambda r: r.get("Title", "").strip() == deal_title.strip(), deal_data)
        with log_output:
            print(f"\u2705 Accepted {current_file} — updated '{deal_title}' (training #{count})")

    elif email_type == "priced":
        price_fields = {
            "Status": "Priced",
            "Priced Date": extraction.get("priced_date") or extraction.get("clo-transactions__Priced Date") or today,
            "Final Pricing Details": extraction.get("final_pricing") or extraction.get("clo-transactions__Final Pricing Details") or "",
        }
        price_fields.update(txn_data)
        store.update_record(
            store.TRANSACTIONS,
            lambda r: r.get("Deal", "").strip() == deal_title.strip() and r.get("Status") != "Priced",
            price_fields,
        )
        if deal_data:
            store.update_record(store.DEALS, lambda r: r.get("Title", "").strip() == deal_title.strip(), deal_data)
        with log_output:
            print(f"\u2705 Accepted {current_file} — priced '{deal_title}' (training #{count})")

    queue.move_to_processed(current_file)
    queue.pop()
    _load_next_email()
def _on_skip(btn):
    if not _gui_state["file"]:
        with log_output:
            print("No email loaded.")
        return
    skipped = queue.skip()
    with log_output:
        print(f"⏭️  Skipped: {skipped}")
    _load_next_email()


def _on_refresh(btn):
    log_output.clear_output()
    with log_output:
        print("🔄 Refreshing queue...")
    _hide_done_prompt()
    _load_next_email()


btn_accept.on_click(_on_accept)
btn_skip.on_click(_on_skip)
btn_refresh.on_click(_on_refresh)
btn_refresh_done.on_click(_on_refresh)

_load_next_email()
display(gui)

In [ ]:
# Force-refresh all widgets from the extraction
extraction = _gui_state.get("extraction", {})
for fname_key, _, ftype in FORM_FIELDS:
   val = extraction.get(fname_key)
   w = field_widgets[fname_key]
   if ftype == "dropdown":
       w.value = val if val in ["announced", "updated", "priced"] else "announced"
   elif ftype == "float":
       w.value = str(val) if val is not None else ""
   else:
       w.value = str(val) if val else ""
print("Refreshed", len(FORM_FIELDS), "fields")
print("Sample values:", {k: extraction.get(k) for k in ["deal_name", "arranger", "email_type"] if extraction.get(k)})

In [ ]:
from IPython.display import display, HTML as DisplayHTML  # <-- Ensure DisplayHTML is imported
# ── Read web GUI HTML and inject comm bridge ──
_html_path = os.path.join(ROOT, "frontend", "index.html")
with open(_html_path, encoding="utf-8") as _f:
    _web_html = _f.read()

# Replace fetch-based api() with Jupyter comm bridge
_web_html = _web_html.replace(
    """async function api(method, path, body) {
  const opts = { method, headers: { 'Content-Type': 'application/json' } };
  if (body) opts.body = JSON.stringify(body);
  const res = await fetch(path, opts);
  return res.json();
}""",
    """
var _clo_comm = null, _clo_pending = {}, _clo_ready = false;
function _initComm() {
  if (_clo_comm) return;
  try {
    var k = Jupyter.notebook ? Jupyter.notebook.kernel : (IPython.notebook ? IPython.notebook.kernel : null);
    if (!k) { console.log('No kernel found'); return; }
    _clo_comm = k.comm_manager.new_comm('clo_gui_api', {});
    _clo_comm.on_msg(function(msg) {
      var d = msg.content.data;
      if (_clo_pending[d.id]) { _clo_pending[d.id](d.result); delete _clo_pending[d.id]; }
    });
    _clo_ready = true;
    console.log('Comm bridge connected');
  } catch(e) { console.log('Comm init error:', e); }
}
async function api(method, path, body) {
  if (!_clo_ready) _initComm();
  if (!_clo_comm) {
    // Fallback to fetch if comm unavailable
    try {
      const opts = { method, headers: { 'Content-Type': 'application/json' } };
      if (body) opts.body = JSON.stringify(body);
      const res = await fetch(path, opts);
      return res.json();
    } catch(e) { return {error: 'No connection'}; }
  }
  return new Promise(function(resolve) {
    var id = Math.random().toString(36).slice(2);
    _clo_pending[id] = resolve;
    _clo_comm.send({id: id, method: method, path: path, body: body || null});
  });
}
setTimeout(_initComm, 50);
"""
    )

# Adjust sizing for notebook embedding
_web_html = _web_html.replace("height: 100vh;", "height: 92vh;")
_web_html = _web_html.replace("overflow: hidden;", "overflow: auto;")
_web_html = _web_html.replace("height: calc(100vh - 50px);", "height: calc(92vh - 50px);")

# Replace shutdown with refresh
_web_html = _web_html.replace(
    '<button class="btn btn-accept" style="background:var(--danger);" onclick="shutdownServer()">Shut Down Server</button>',
    '<button class="btn btn-accept" onclick="dismissShutdown();loadNext();">Refresh Queue</button>'
)
_web_html = _web_html.replace(
    'The email queue is empty. You can shut down the server or keep it running to wait for new emails.',
    'The email queue is empty. Drop more .txt files into data/inbox/ and click Refresh.'
)

# Display directly as HTML output (NOT in an iframe — avoids sandbox/comm issues)
display(DisplayHTML(
    f'<div id="clo-gui-wrapper" style="width:100%;height:92vh;border:1px solid #2a3a5c;border-radius:8px;overflow:hidden;">'
    f'{_web_html}'
    f'</div>'
) )
print("✅ GUI loaded — if fields don't populate, run this cell again after Cell 1")

UnicodeDecodeError: 'charmap' codec can't decode byte 0x8f in position 11685: character maps to <undefined>

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 3 (OPTIONAL): LAUNCH WEB UI — Start server & open browser
# ═══════════════════════════════════════════════════════════════════════
# Only use this if you WANT the browser-based UI instead of the notebook
# GUI above. Requires port 8000 to be accessible (may be blocked by
# corporate firewalls). Cell 2 above does the same thing without a server.
# ═══════════════════════════════════════════════════════════════════════

import threading, time, webbrowser, uvicorn
from backend.app import app as fastapi_app

_server_thread = None
_server_instance = None

def _start_server():
    global _server_instance
    server_config = uvicorn.Config(
        fastapi_app,
        host="127.0.0.1:",
        port=config.PORT,
        log_level="info",
    )
    _server_instance = uvicorn.Server(server_config)
    fastapi_app.state._server_handle = _server_instance
    _server_instance.run()

import socket
def _port_in_use(port):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        return s.connect_ex(("127.0.0.1", port)) == 0

if _port_in_use(config.PORT):
    print(f"✅ Server already running on http://localhost:{config.PORT}")
else:
    _server_thread = threading.Thread(target=_start_server, daemon=True)
    _server_thread.start()
    for _ in range(20):
        if _port_in_use(config.PORT):
            break
        time.sleep(0.25)
    print(f"✅ Server started on http://localhost:{config.PORT}")

webbrowser.open(f"http://localhost:{config.PORT}")
print("🌐 Browser opened — use the web UI to process emails.")

In [ ]:
fname = queue.peek()
if fname:
    html = queue.read_file(fname)
    print("First 500 Chars of First Email:")
    print(html[:500])  
    print("\n--Extraction Result--")
    print(_gui_state.get("extraction", {}))

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 4: VIEW ALL SAVED DATA
# ═══════════════════════════════════════════════════════════════════════
# Run anytime to see what's been saved to the JSON stores.
# ═══════════════════════════════════════════════════════════════════════

display(Markdown("### Deals"))
deals = store.read_store(store.DEALS)
if deals:
    for d in deals:
        print(f"  {d.get('deal_name')} | {d.get('collateral_manager_short')} | {d.get('arranger')} | ${d.get('target_par', '?')}MM")
else:
    print("  (none)")

display(Markdown("### Transactions"))
txns = store.read_store(store.TRANSACTIONS)
if txns:
    for t in txns:
        print(f"  {t.get('deal_name')} | {t.get('status')} | {t.get('transaction_type')} | {t.get('announced_date', '')}")
else:
    print("  (none)")

display(Markdown("### Deal Orders"))
orders = store.read_store(store.DEAL_ORDERS)
if orders:
    for o in orders:
        print(f"  {o.get('deal_name')} | {o.get('tranche')} | ${o.get('allocation', '?')}MM")
else:
    print("  (none)")

display(Markdown("### Managers"))
mgrs = store.read_store(store.MANAGERS)
if mgrs:
    for m in mgrs:
        print(f"  {m.get('short_name')} | {m.get('legal_entity')}")
else:
    print("  (none)")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 4: TRAINING STATUS
# ═══════════════════════════════════════════════════════════════════════

import os
training_path = config.TRAINING_DATA
if os.path.exists(training_path):
    with open(training_path, "r") as f:
        count = sum(1 for _ in f)
    print(f"📊 Training examples collected: {count}")
    if count < 5:
        print(f"   Need at least 5 to fine-tune. Keep processing emails!")
    else:
        print(f"   ✅ Ready to fine-tune. Run Cell 5.")
else:
    print("No training data yet. Accept some emails in the GUI first.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 5: FINE-TUNE THE MODEL (LoRA)
# ═══════════════════════════════════════════════════════════════════════
# After training, restart the kernel and re-run Cells 1-2.
# ═══════════════════════════════════════════════════════════════════════

from backend.finetune import run_finetune

print("Starting fine-tuning...")
print("=" * 60)
adapter_path = run_finetune(num_epochs=3, learning_rate=2e-4, lora_r=16)
if adapter_path:
    print("=" * 60)
    print(f"\n🎉 Done! Restart kernel and re-run Cells 1-2 to use the improved model.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 6: RESET (clear all saved data)
# ═══════════════════════════════════════════════════════════════════════
# Uncomment and run to clear everything. CAREFUL — this is irreversible.
# ═══════════════════════════════════════════════════════════════════════

# for s in store.ALL_STORES:
#     store.clear_store(s)
# print("All data stores cleared.")

# import os
# if os.path.exists(config.TRAINING_DATA):
#     os.remove(config.TRAINING_DATA)
#     print("Training data cleared.")